In [1]:
# %% Cell 1: imports
import sys, torch, numpy as np
sys.path.append('..'); sys.path.append('../..')

from common.seed import set_seed
from common.io_utils import save_results, load_results
from backbones.backbones import get_resnet50, get_vit, get_clip, extract_all_features
from img_prep.make_subset import load_stl10
from img_prep.transforms import (resnet_normalize, vit_normalize, clip_normalize,
                                   grayscale_transform, hue_rotate_transform)
from evaluations.evaluate_bias import evaluate_head_with_preds, prediction_consistency, clip_zero_shot_eval
import open_clip, torch.nn as nn

set_seed(6304)

In [2]:
# %% Cell 2: load subset + clean predictions (the reference point)
subset_ids = load_results("../results/subset_ids.json")
test_idx = subset_ids["test_subset_indices"]
clean_preds = load_results("../results/clean_predictions.json")["predictions"]
clean_baseline = load_results("../results/clean_baseline.json")["models"]

test_ds = load_stl10(split='test')
class_names = ['airplane','bird','car','cat','deer','dog','horse','monkey','ship','truck']

In [3]:
# %% Cell 3: load backbones + reload trained heads from checkpoint
resnet, _ = get_resnet50()
vit, _ = get_vit()
clip_model, _ = get_clip()
tokenizer = open_clip.get_tokenizer('ViT-B-32')

def load_head(name, in_dim, num_classes=10):
    head = nn.Linear(in_dim, num_classes)
    head.load_state_dict(torch.load(f"../../datasets/cache/{name}_head.pth"))
    head.eval()
    return head

heads = {
    "resnet50": load_head("resnet50", 2048),
    "vit_b16": load_head("vit_b16", 768),
    "clip_head": load_head("clip_head", 512),
}

c:\Users\javai\anaconda3\Lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


In [ ]:
# %% Cell 4: run both interventions through both models, compare to clean
interventions = {"grayscale": grayscale_transform, "hue_rotate": lambda x: hue_rotate_transform(x, 0.5)}
backbones_info = [("resnet50", resnet, resnet_normalize, "resnet_or_vit"),
                   ("vit_b16", vit, vit_normalize, "resnet_or_vit"),
                   ("clip", clip_model, clip_normalize, "clip")]

results = {"step": "color_bias", "seed": 6304, "interventions": {}}

for interv_name, interv_fn in interventions.items():
    results["interventions"][interv_name] = {}
    for name, model, norm_fn, mtype in backbones_info:
        feats, labels = extract_all_features(model, test_ds, test_idx, norm_fn, model_type=mtype, intervention_fn=interv_fn)

        if name == "clip":
            # both the trained head AND zero-shot for CLIP
            head_metrics, head_preds = evaluate_head_with_preds(heads["clip_head"], feats, labels)
            results["interventions"][interv_name]["clip_head"] = {
                **head_metrics,
                "acc_change": head_metrics["top1_acc"] - clean_baseline["clip_head"]["top1_acc"],
                "consistency": prediction_consistency(np.array(clean_preds["clip_head"]), head_preds)
            }
            imgs = torch.stack([interv_fn(x) for x in [__import__('img_prep.transforms', fromlist=['to_common_224']).to_common_224(img) for img, _ in torch.utils.data.Subset(test_ds, test_idx)]])
            imgs = torch.stack([clip_normalize(x) for x in imgs])
            zs_metrics, zs_preds = clip_zero_shot_eval(clip_model, tokenizer, imgs, labels, class_names)
            results["interventions"][interv_name]["clip_zeroshot"] = {
                **zs_metrics,
                "acc_change": zs_metrics["top1_acc"] - clean_baseline["clip_zeroshot"]["top1_acc"],
                "consistency": prediction_consistency(np.array(clean_preds["clip_zeroshot"]), zs_preds)
            }
        else:
            metrics, preds = evaluate_head_with_preds(heads[name], feats, labels)
            results["interventions"][interv_name][name] = {
                **metrics,
                "acc_change": metrics["top1_acc"] - clean_baseline[name]["top1_acc"],
                "consistency": prediction_consistency(np.array(clean_preds[name]), preds)
            }
        print(interv_name, name, results["interventions"][interv_name][name if name!="clip" else "clip_head"])

TypeError: extract_all_features() got an unexpected keyword argument 'intervention_fn'

In [ ]:
# %% Cell 5: save
save_results(results, "../results/color_bias.json")
print(results)